# Notebook to process lysosomal and mitochondrial morphology from cellprofiler features - this time from a merged cell CSV aready created
Inputs Required
 - Database with cellprofiler outputs
   - Mainly use the Per_Cell table for per-cell features
     - Intensity
     - AreaShape features
     - Ratio of organelle area to total cell area
     - Texture features
     - Granularity features
     - Radial intensity distribution about nucleus
   - May additionally want to use mitochdondria or lysosomes.csv if analyzing these specifically 
     - Note that lysosome segmentation is not perfect (but i'm proud of it)
 - Metadata CSV representing 96-well platemap
   - Long-form table containing passage number, staining conditons, treatments, etc
   - Produced automatically from an 8x12 table with the 96Well_PlateMap code
Outputs
- graphs for individual features
- clustering
## Set your paths here

In [ ]:
#Imports
import os
from pathlib import Path
import numpy as np
import pandas as pd
import sqlite3
#plotting
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from scipy import stats

from plate_information import *
from plate_preprocessing import *
from mitolyso_plot_functions import *

## Import the big csv

In [ ]:
#import from a giant csv
#csvpath = "/Volumes/AllieS/Morphology_data/"
csvpath = '/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs/'

filename = 'total_combined_cell.csv'
#filename = "total_combined_cell_borders_excluded.csv"
combined_cell_df_mitolyso = pd.read_csv(os.path.join(csvpath, filename))
# filter_df = enforce_objects_one_to_one(combined_cell_df_mitolyso)
display(combined_cell_df_mitolyso.shape)
#print(cell_df.shape, " ", filter_df.shape)

#combined_cell_df_mitolyso_merged = plate_df_setup(curr_plates, curr_plate_datafolders, parent_dir, ['Cell.csv', 'Nuclei.csv','MergedMitoPerCell.csv','MergedLysoPerCell.csv'])

filename_borders_excluded = "total_combined_cell_borders_excluded.csv"
combined_cell_df_mitolyso_borders_excluded = pd.read_csv(os.path.join(csvpath, filename_borders_excluded))

In [ ]:
unique_combinations = combined_cell_df_mitolyso[['Replicate_Number', 'PassageNumber']].drop_duplicates()

combos = list(unique_combinations.itertuples(index=False, name=None))
display(sorted(combos))
display(combined_cell_df_mitolyso)

## Debugging functions


### Filter out the poorly segmented cells and rename the columns so I don't have to change the old code

In [ ]:
def enforce_objects_one_to_one(
    df,
    parent_obj="Cell",
    child_obj="Nuclei",
    parent_colname="Cell_AreaShape_Area",
    child_colname="Cell_Mean_Nuclei_AreaShape_Area",
):
    """apply filters based on the number of nuclei to remove to enforce a 1:1 cell-nucleus relationship:
    - Cells that have less or more than one nucleus
    - Poorly segmented cells where the cell area is smaller than the nuclei area
    - Image is not flagged as empty by CellProfiler
    Can also use other objects instead of nuclei

    Args:
        df (DataFrame): The parent dataframe
        child_obj (str, optional): _description_. Defaults to "Nuclei".
        parent_colname (str, optional): _description_. Defaults to "Cell_AreaShape_Area".
        child_colname (str, optional): _description_. Defaults to "Cell_Mean_Nuclei_AreaShape_Area".

    Returns:
        DataFrame: the filtered dataframe with the removed objects
    """
    if child_obj == "Nuclei" and parent_obj == "Cell":
        try:
            not_empty_df = df[df["Metadata_EmptyImage_Cells"] == 0]
            normal_cells = not_empty_df[
                not_empty_df["Cell_Classify_one_nuc"] == 1
            ]  # one nucleus only
        except KeyError as e:
            print(f"KeyError {e}; skipping emptyimage")
            normal_cells = df[df["Cell_Classify_one_nuc"] == 1]
        # remove if my cell area is bigger than nuclear area
        size_filtered_cells = normal_cells[
            normal_cells[parent_colname] > normal_cells[child_colname]
        ]
        final_df = size_filtered_cells.reset_index(drop=True)
        return final_df
    else:
        # just do the size excludion
        not_empty_df = df[df["Metadata_EmptyImage_Cells"] == 0]
        size_filtered_cells = not_empty_df[
            not_empty_df[parent_colname] > not_empty_df[child_colname]
        ]
        return size_filtered_cells


def filter_nuclei_outisde_bbox(overlapped_df, nuc_prefix = "Nuclei", cell_prefix = ""):
    
    df = overlapped_df.copy()
    #Get the coordinate of the cell and nuclei objects
    #Note- make sure the cells and nuclei are 1:1 or this will fail
    nuc_min_x, nuc_min_y, nuc_max_x, nuc_max_y = (
        df[f"{nuc_prefix}_AreaShape_BoundingBoxMinimum_X"],
        df[f"{nuc_prefix}_AreaShape_BoundingBoxMinimum_Y"],
        df[f"{nuc_prefix}_AreaShape_BoundingBoxMaximum_X"],
        df[f"{nuc_prefix}_AreaShape_BoundingBoxMaximum_Y"]
    )
    cell_min_x, cell_min_y, cell_max_x, cell_max_y = (
        df[f"{cell_prefix}AreaShape_BoundingBoxMinimum_X"],
        df[f"{cell_prefix}AreaShape_BoundingBoxMinimum_Y"],
        df[f"{cell_prefix}AreaShape_BoundingBoxMaximum_X"],
        df[f"{cell_prefix}AreaShape_BoundingBoxMaximum_Y"]
    )
    #now get two boolean series that match the conditions
    df_is_contained_x = (nuc_min_x >= cell_min_x) & (nuc_max_x <= cell_max_x)
    df_is_contained_y = (nuc_min_y >= cell_min_y) & (nuc_max_y <= cell_max_y)
    #and then we get the subset of the dataframe where both are true
    df_is_contained_xy = df[df_is_contained_x & df_is_contained_y]
    
    final_df = df_is_contained_xy.reset_index(drop=True)
    return final_df


def filter_out_empty_compartment_from_cells(df, organelle, prefix=""):
    """Remove rows from a dataframe where a parent "cell" object doesn't have any child objects of {organelle}

    Args:
        df (DataFrame): _description_
        organelle (str): the organelle in plural. Typically "Mitochondria or Lysosomes (or Nuclei)
        prefix (str, optional): _description_. Defaults to "".

    Returns:
        Dataframe: _description_
    """    
    organelle = organelle.title() 
    this_df = df.copy()
    atleast_one_df = this_df[this_df[f"{prefix}Children_{organelle}_Count"] > 1].reset_index(drop=True)  
    return atleast_one_df

filtered_df = enforce_objects_one_to_one(combined_cell_df_mitolyso)#, child_colname="Nuclei_AreaShape_Area")
filtered_df.columns = filtered_df.columns.str.replace(r'^Cell_', '', regex=True)
filter_df_copy = filtered_df.copy()
#also rename nuclei cols in the filtered df when you made the df one-to-one
filter_df_copy.columns = filter_df_copy.columns.str.replace(
    r"^Mean_Nuclei_", "Nuclei_", regex=True
)
filter_df_copy["Cell_Nuclei_Area_Ratio"] = filter_df_copy[
    "Nuclei_Area_Ratio"
] #dupe column with the old name back


def filter_out_images_with_n_cells(df, n, image_count_col = "Image_Count_Cell", prefix=""):
    filter_df = df.copy()
    filter_df = filter_df[image_count_col] > n
    final_df = filter_df.reset_index(drop=True)
    return final_df
    
nona_df = filtered_df.dropna()

#filter out the cells wo compartments
filter_df_1 = filter_out_empty_compartment_from_cells(filter_df_copy, "mitochondria")
filter_df_2 = filter_out_empty_compartment_from_cells(filter_df_1, "lysosomes")
filter_df_morethanonecell = filter_df_2.copy()[filter_df_2["Image_Count_Cell"] > 1]
filter_df_outbbox = filter_nuclei_outisde_bbox(filter_df_morethanonecell)

final_filtered_df = filter_df_outbbox.copy()#filter_df_2.copy()


print(
    f"OG df: {combined_cell_df_mitolyso.shape}, Filtered df: {filtered_df.shape}, dropped na: {nona_df.shape}, removing empty organelles: {filter_df_2.shape}, more than one cell: {filter_df_morethanonecell.shape}, no nuc outside cells: {filter_df_outbbox.shape}"
)

In [ ]:
final_filtered_df_2 = final_filtered_df[final_filtered_df["Classify_Saturated"] < 1]
print(final_filtered_df_2.shape)

saturated_cells = final_filtered_df[final_filtered_df["Classify_Saturated"] == 1]

In [ ]:
#Simple column search function
def search_column_name(df,query=""):
    """_summary_

    Args:
        df (DataFrame): _description_
        query (str, optional): your search query. Defaults to "".
    Return:

    """    
    query_cols = [col for col in df.columns if query in col]
    print(f"Query: {query}")
    for col in query_cols:
        print(f"    {col}")
    return query_cols
# Display columns containing 'metadata'

colnames = search_column_name(final_filtered_df,"Classify")



In [ ]:
#display(combined_cell_df_mitolyso[['Cell_AreaShape_Area', 'Cell_Children_Mitochondria_Count','Cell_Mean_Mitochondria_AreaShape_Area']])
display(final_filtered_df[['AreaShape_Area', 'Children_Mitochondria_Count','Mean_Mitochondria_AreaShape_Area']])


# Define the cell features


In [ ]:
# Add extra columns
def proportion_area_occupied_per_cell(df, compartment):
    # proportion of area occupied = children * mean organelle area / cell area
    colname = "Total_Area_Proportion_" + compartment + "_Per_Cell"

    # children = 'Children_' + compartment + '_Count'
    # mean_organelle_area = 'Mean_'+ compartment + '_AreaShape_Area'
    organelle_area = compartment + "_AreaShape_Area"
    cell_area = "AreaShape_Area"
    # df[colname] = df.apply(lambda x: (x[children] * x[mean_organelle_area]) / x[cell_area], axis=1)
    df[colname] = df.apply(lambda x: (x[organelle_area]) / x[cell_area], axis=1)
    return df[colname]


def mean_intensity_per_compartment_per_cell(df, compartment, name, tag, math=None):
    # Calculate the mean intensity of each compartment per cell
    # mean_intesity_per_compartment = integrated / (children*mean_area)
    colname = f"Mean_Intensity_Per_{compartment} Per_Cell"
    integrated = "Intensity_IntegratedIntensity_" + tag
    # children = 'Children_' + compartment + '_Count'
    # mean_area = 'Mean_'+ compartment + '_AreaShape_Area'
    total_organelle_area = name + "_AreaShape_Area"
    total_organelle_area = math if math is not None else total_organelle_area

    df[colname] = df.apply(lambda x: x[integrated] / x[total_organelle_area], axis=1)

    return df[colname]

# df["Mean_Mitochondria_Area_PerCell_Ratio"]  = proportion_area_occupied_per_cell(combined_cell_df_mitolyso_merged, "MergedMitoPerCell")
# df["Mean_Lysosomes_Area_PerCell_Ratio"]  = proportion_area_occupied_per_cell(df, "MergedLysoPerCell")

# df["MeanIntensity_Lysosomes_PerCell_Ratio"] = mean_intesity_per_compartment_per_cell(combined_cell_df_mitolyso_merged, "Lysosomes", "MergedLysoPerCell", "LAMP1")
# df["MeanIntensity_Mitochondria_PerCell_Ratio"] = mean_intesity_per_compartment_per_cell(df, "Mitochondria", "MergedMitoPerCell", "MitoTracker")

# Calculate the total area occupied by mitochondria and lysosomes per cell
def calculate_corrected_features(full_df):
    """_summary_

    Args:
        full_df (DataFrame): _description_

    Returns:
        df (DataFrame): the df with all the feature calcs
    """    
    df = full_df.copy()
    df["Math_Total_Mitochondria_AreaShape_Area_PerCell"] = (
        df["Children_Mitochondria_Count"] * df["Mean_Mitochondria_AreaShape_Area"]
    )
    df["Math_Total_Lysosomes_AreaShape_Area_PerCell"] = (
        df["Children_Lysosomes_Count"] * df["Mean_Lysosomes_AreaShape_Area"]
    )

    # Total intensity per cell based on ingegrated instensity if I don't already have the merged area
    df["Mean_Intensity_Per_Lysosomes_PerCell_Area"] = (
        mean_intensity_per_compartment_per_cell(
            df,
            "Lysosomes",
            "MergedLysoPerCell",
            "LAMP1",
            math="Math_Total_Lysosomes_AreaShape_Area_PerCell",

        )
    )
    df["Mean_Intensity_Per_Mitochondria_PerCell_Area"] = (
        mean_intensity_per_compartment_per_cell(
            df,
            "Mitochondria",
            "MergedMitoPerCell",
            "MitoTracker",
            math="Math_Total_Mitochondria_AreaShape_Area_PerCell",
        )
    )
    #same thing but for medians
    df["Median_Intensity_Per_Lysosomes_PerCell_Area"] = (
        df["Children_Lysosomes_Count"]
        * df["Mean_Lysosomes_Intensity_MeanIntensity_LAMP1"]
    )
    df["Median_Intensity_Per_Mitochondria_PerCell_Area"] = (
        df["Children_Mitochondria_Count"]
        * df["Mean_Mitochondria_Intensity_MeanIntensity_MitoTracker"]
    )

    # Corrected mitochondria and lysosomes counts per cell area (density)
    df["Density_Children_Mitochondria_Count_PerCell_Area"] = (
        df["Children_Mitochondria_Count"] / df["AreaShape_Area"]
    )
    df["Density_Children_Lysosomes_Count_PerCell_Area"] = (
        df["Children_Lysosomes_Count"] / df["AreaShape_Area"]
    )

    # organelle area fractions per cell ratio
    df["OccupiedAreaFraction_Mitochondria_PerCell_Area"] = (
        df["Math_Total_Mitochondria_AreaShape_Area_PerCell"] / df["AreaShape_Area"]
    )
    df["OccupiedAreaFraction_Lysosomes_PerCell_Area"] = (
        df["Math_Total_Lysosomes_AreaShape_Area_PerCell"] / df["AreaShape_Area"]
    )
    
    # Compartment diameter ratios
    df["Mean_Lysosomes_DiameterRatio_PerCell"] = (
        df["Mean_Lysosomes_AreaShape_MaxFeretDiameter"]
        / df["Mean_Lysosomes_AreaShape_MinFeretDiameter"]
    )
    df["Mean_Mitochondria_DiameterRatio_PerCell"] = (
        df["Mean_Mitochondria_AreaShape_MaxFeretDiameter"]
        / df["Mean_Mitochondria_AreaShape_MinFeretDiameter"]
    )
    df["Median_Mitochondria_DiameterRatio_PerCell"] = (
        df["Median_Mitochondria_AreaShape_MaxFeretDiameter"]
        / df["Median_Mitochondria_AreaShape_MinFeretDiameter"]
    )
    df["Median_Lysosomes_DiameterRatio_PerCell"] = (
        df["Median_Lysosomes_AreaShape_MaxFeretDiameter"]
        / df["Median_Lysosomes_AreaShape_MinFeretDiameter"]
    )

    # mean and median area per organelle per cell area
    df["Mean_Mitochondria_Area_PerCell_Area"] = (
        df["Mean_Mitochondria_AreaShape_Area"] / df["AreaShape_Area"]
    )
    df["Mean_Lysosomes_Area_PerCell_Area"] = (
        df["Mean_Lysosomes_AreaShape_Area"] / df["AreaShape_Area"]
    )
    df["Median_Mitochondria_Area_PerCell_Area"] = (
        df["Median_Mitochondria_AreaShape_Area"] / df["AreaShape_Area"]
    )
    df["Median_Lysosomes_Area_PerCell_Area"] = (
        df["Median_Lysosomes_AreaShape_Area"] / df["AreaShape_Area"]
    )

    # mitolyso related
    df["Children_Lysosomes_Mitochondria_Ratio"] = (
        df["Children_Lysosomes_Count"] / df["Children_Mitochondria_Count"]
    )
    df["Density_Lysosomes_Mitochondria_Ratio"] = (
        df["OccupiedAreaFraction_Lysosomes_PerCell_Area"]
        / df["OccupiedAreaFraction_Mitochondria_PerCell_Area"]
    )
    df["Area_Lysosomes_Mitochondria_Ratio"] = (
        df["Math_Total_Lysosomes_AreaShape_Area_PerCell"]
        / df["Math_Total_Mitochondria_AreaShape_Area_PerCell"]
    )


    # Ratio of centroid distance to minimum distance for mitochondria and lysosomes
    df["Mean_Mitochondria_Distance_Centroid_Minimum_Ratio"] = (
        df["Mean_Mitochondria_Distance_Centroid_Nuclei"]
        / df["Mean_Mitochondria_Distance_Minimum_Nuclei"]
    )
    df["Mean_Lysosomes_Distance_Centroid_Minimum_Ratio"] = (
        df["Mean_Lysosomes_Distance_Centroid_Nuclei"]
        / df["Mean_Lysosomes_Distance_Minimum_Nuclei"]
    )

    # transform the mitoends - number of ends times the mean to get total per cell
    df["MitoEnds_Math_Total_NumberBranchEnds_MitoSkeleton"] = (
        df["Children_MitoEnds_Count"]
        * df["Mean_MitoEnds_ObjectSkeleton_NumberBranchEnds_MitoSkeleton"]
    )
    df["MitoEnds_Math_Total_NumberNonTrunkBranches_MitoSkeleton"] = (
        df["Children_MitoEnds_Count"]
        * df["Mean_MitoEnds_ObjectSkeleton_NumberNonTrunkBranches_MtSkltn"]
    )
    df["MitoEnds_Math_Total_NumberTrunks_MitoSkeleton"] = (
        df["Children_MitoEnds_Count"]
        * df["Mean_MitoEnds_ObjectSkeleton_NumberTrunks_MitoSkeleton"]
    )
    df["MitoEnds_Math_TotalObjectSkeltnLngth_MitoSkeleton_PerCell"] = (
        df["Children_MitoEnds_Count"]
        * df["Mean_MitoEnds_ObjectSkeleton_TotalObjectSkeltnLngth_MtSkltn"]
    )
    df["MitoEnds_Total_ObjectSkeltnLngth_MitoSkeleton_PerCell_Area"] = (
        df["MitoEnds_Math_TotalObjectSkeltnLngth_MitoSkeleton_PerCell"]
        / df["AreaShape_Area"]
    )
    return df

combined_cell_df_mitolyso_merged = calculate_corrected_features(final_filtered_df)
#display(combined_cell_df_mitolyso_merged)
display(
    combined_cell_df_mitolyso_merged[
        [
            "Children_Mitochondria_Count",
            "Mean_Mitochondria_AreaShape_Area",
            "Math_Total_Mitochondria_AreaShape_Area_PerCell",
            "Median_Mitochondria_Area_PerCell_Area",
            "Mean_Mitochondria_Area_PerCell_Area",
            "OccupiedAreaFraction_Mitochondria_PerCell_Area",
            "Mean_Lysosomes_DiameterRatio_PerCell",
            "Mean_Mitochondria_Distance_Centroid_Minimum_Ratio",
            "Density_Children_Mitochondria_Count_PerCell_Area",
            "Mean_Intensity_Per_Mitochondria_PerCell_Area",
        ]
    ]
)


# Note: use the "Corr_" flag to get the corrected values
# display(combined_cell_df_mitolyso_merged[["Passage Group","Mean_Lysosomes_Intensity_MeanIntensity_LAMP1","MeanIntensity_Mitochondria_PerCell_Ratio","MeanIntensity_Lysosomes_PerCell_Ratio","Mean_Lysosomes_Area_PerCell_Ratio","Mean_Mitochondria_Area_PerCell_Ratio","Math_Total_Mitochondria_AreaShape_Area_PerCell","Math_Total_Lysosomes_AreaShape_Area_PerCell"]])

In [ ]:
# file_path = '/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/Cellcsv_columns.txt'
# pairs_samples = [('P6-8', 'P9-10'), ('P6-8', 'P11-13'), ('P6-8', 'P14-16'), ('P6-8', 'P17-18'),
# ('P6-8', 'P20-21'), ('P6-8', 'P22-24'), ('P9-10', 'P11-13'), ('P9-10', 'P14-16'), ('P9-10', 'P17-18'),
# ('P9-10', 'P20-21'), ('P9-10', 'P22-24'), ('P11-13', 'P14-16'), ('P11-13', 'P17-18'), ('P11-13',
# 'P20-21'), ('P11-13', 'P22-24'), ('P14-16', 'P17-18'), ('P14-16', 'P20-21'), ('P14-16', 'P22-24'),
# ('P17-18', 'P20-21'), ('P17-18', 'P22-24'), ('P20-21', 'P22-24')]
columns_list = define_cell_features(combined_cell_df_mitolyso_merged)


def define_cell_features(df):
    # Get the columns of the dataframe
    columns_list = df.columns.tolist()
    columns_list = [
        col
        for col in columns_list
        if "Metadata" not in col
        and "FileName" not in col
        and "PathName" not in col
        and pd.api.types.is_numeric_dtype(df[col])
    ]
    old_columns_list = columns_list = [
        col
        for col in columns_list
        if "Metadata" not in col and "FileName" not in col and "PathName" not in col
    ]
    print(
        "Original columns:",
        len(old_columns_list),
        "Filtered columns:",
        len(columns_list),
    )
    return columns_list


mito_features = make_feature_dict(
    [
        col
        for col in columns_list
        if ("Mito" in col or "Mitochondria" in col)
        and ("DAPI" not in col and "LAMP1" not in col and "Frame" not in col)
        and not (col.startswith("Nuclei_"))
    ]
)
lyso_features = make_feature_dict(
    [
        col
        for col in columns_list
        if ("Lysosome" in col or "LAMP1" in col or "Lyso" in col)
        and ("DAPI" not in col and "Mito" not in col and "Frame" not in col)
        and not(col.startswith("Nuclei_"))
    ]
)
nuc_features = make_feature_dict(
    [
        col
        for col in columns_list
        if ("Nuc" in col or "DAPI" in col)
        and ("MitoTracker" not in col and "LAMP1" not in col and "Frame" not in col)
    ]
)
cell_features = make_feature_dict(
    [
        col
        for col in columns_list
        if "AreaShape" in col
        and "Mito" not in col
        and "Lyso" not in col
        and "LAMP1" not in col
        and "Nuc" not in col
        and "DAPI" not in col
        and "Metadata" not in col
        and "FileName" not in col
        and "PathName" not in col
    ]
)
print(columns_list)


## Feature lists here:

In [ ]:
feature_dicts = [mito_features, lyso_features, nuc_features, cell_features]
feature_names = ["Mitochondria Features", "Lysosome Features", "Nucleus Features", "Cell Features"]

# Define the output file path
output_file_path = 'allfeatures_file.md'

# Open the file in write mode
with open(output_file_path, 'w') as file:
    for feature_name, feature_dict in zip(feature_names, feature_dicts):
        file.write(f"# {feature_name}\n\n")
        for feature_type, features in feature_dict.items():
            file.write(f"## {feature_type.capitalize()}\n")
            for feature in features:
                file.write(f"\"{feature}\",\n")
            file.write("\n")
            
print(f"List has been written to {output_file_path}")


## Check if there are any mixed types hiding out

In [ ]:
#Debug to check for mixed types in columns
for col in combined_cell_df_mitolyso_merged.columns:
    types = combined_cell_df_mitolyso_merged[col].apply(type).value_counts()
    if len(types) > 1:
        print(f"Column '{col}' has mixed types: {types}")

# Functions for Data Analysis
calulcate normalizations, remove extreme left outliers, etc

## Normalize features to control (Passage 6-8)

In [ ]:
curr_plates = ["20240313_rep01","20240326_rep02","20241018_rep03", "20241112_rep04", "20250328_rep05", "20250410_rep06", "20250501_rep07"] #"20240313_rep01_output","20240326_rep02_output"

#print(combined_cell_df_mitolyso_merged.iloc["*","3f9f05979b4d963d3a416940cd146486c35afd73ff8824950945744285caf345b27996985b25de4f280bbe6380ecd3bdb27996985b25de4f280bbe6380ecd3bdaac39e649e9e480b6aa7e043c7ae0740aac39e649e9e480b6aa7e043c7ae0740ebad8781c113930a511fb1e0b33a5ae1ebad8781c113930a511fb1e0b33a5ae1a6ca8186980a3310d2f7e3d35512fcb3a6ca8186980a3310d2f7e3d35512fcb3c3b031fa1953fe9aed603e94b738cd8dc3b031fa1953fe9aed603e94b738cd8df08d58956a263e00e263dbe462b2b8781c992d8f3dd23b778820570de386ab371c992d8f3dd23b778820570de386ab371c992d8f3dd23b778820570de386ab371c992d8f3dd23b778820570de386ab37b4de0c127359612d367afa1651150c8fb4de0c127359612d367afa1651150c8fb4de0c127359612d367afa1651150c8f6b0e15f67abc6df786bf563bc136c3676b0e15f67abc6df786bf563bc136c3676b0e15f67abc6df786bf563bc136c3674a8539031734babcfd06ff30311dbf064a8539031734babcfd06ff30311dbf065918ba3f0a658eec695b9622d9b20547be25a504ff9c89fe118269e9bacab62e3f51d4573daea38cf97711a9273f5ea5155a754ccadd310a7a08caf6dfb4b2b8ca83e03b639c81a341d51d0c7ddff9ea060bc1e34e6f5eb7e3e6b302803a27ca060bc1e34e6f5eb7e3e6b302803a27ca8f3d782190481b6e951c89cc6d8ec2278f3d782190481b6e951c89cc6d8ec2278f3d782190481b6e951c89cc6d8ec2278f3d782190481b6e951c89cc6d8ec2278f3d782190481b6e951c89cc6d8ec22703397b32121e5f601a63e09d3aac334803397b32121e5f601a63e09d3aac33483d056f026c80c5ae4e7432aa1aa370bcadb33dceaa2f3a88b5f8bd16cc89e0c8fe3594b08af4a6536ac08d8b57a23de7e4f270ab1aed64ab67cc2525f78a7b20afffb56f4b746063b8425c7e265e2eaba5ae3e58d97a40077c5fdf1b46d9ba10a5ae3e58d97a40077c5fdf1b46d9ba10a5ae3e58d97a40077c5fdf1b46d9ba10a5ae3e58d97a40077c5fdf1b46d9ba105f4f978e715ee311af26d2fdabe1c7135f4f978e715ee311af26d2fdabe1c713e1865dce5f347eb6f67db5a65479d1b8e1865dce5f347eb6f67db5a65479d1b8e1865dce5f347eb6f67db5a65479d1b8e1865dce5f347eb6f67db5a65479d1b8e1865dce5f347eb6f67db5a65479d1b869980026743a730b2867acfd8ad94b1169980026743a730b2867acfd8ad94b11"])
def normalize_to_control(df, feature, norm_column = 'AgeGroup'):
    '''
    Normalize a feature to the control group (AgeGroup = 0) for each plate.
    Args:
        df (DataFrame): The DataFrame containing the feature to be normalized.
        feature (str): The name of the feature column to normalize.
        norm_column (str): The column used to identify the control group (default is 'AgeGroup').'`
    Returns:
        Series: A Series containing the normalized feature values.
    '''
    # Take the t0 df - lowest passage data point
    t0_df = df[df[norm_column]==0]
    treatment_df = df[[feature, norm_column]].copy()

    #calculate the mean
    mean_zero = t0_df[feature].mean()
    # Check for non-numeric values
    if not pd.api.types.is_numeric_dtype(treatment_df[feature]):
        print(f"[normalize_to_control] WARNING: {feature} is not numeric!")
    #now update the column to have all rows dividied by the mean of group 0
    treatment_df["norm_" + feature] = treatment_df[feature] / mean_zero  
    #return the normalized feature columnn
    return treatment_df["norm_" + feature]

def normalize_features(df, feature_list):
    '''
    Normalize the features in the DataFrame to the control (age group 0) for each plate.
    Args:
        df (DataFrame): The DataFrame containing the features to be normalized.
        feature_list (list): A list of feature column names to normalize.
    Returns:
        DataFrame: A DataFrame with normalized features for each plate.
    '''
    # Normalize the features to the control (age group 0) for each plate
    norm_df = df.copy()
    for feature in feature_list:
        #print('Normalizing feature: ', feature, '...', norm_df[feature].values[0])
        norm_df[feature] = normalize_to_control(df, feature)
        #print('Normalized feature: ', feature, '...', norm_df[feature].values[0])
    return norm_df


def apply_feature_normalization(df, feature_dict, curr_plates):
    '''
    Apply feature normalization to the DataFrame for each plate in a list of plates.
    Args:
        df (DataFrame): The DataFrame containing the features to be normalized.
        feature_dict (dict): A dictionary containing lists of feature columns to normalize.
        curr_plates (list): A list of plate names to apply normalization to.
    Returns:
        DataFrame: A DataFrame with normalized features for each plate.
    '''
    # Normalize the features to the control (age group 0) for each plate
    norm_cell_df = df.copy()
    for plate in curr_plates:
        curr_plate_df = norm_cell_df[norm_cell_df['Metadata_Plate'] == plate].copy()
        for feature_type in feature_dict:
            #get the normalized features, locate the corresponding features on the plate, and replace them on that plate to the plate
            curr_plate_features_df = normalize_features(curr_plate_df, feature_dict[feature_type])
            curr_plate_df.loc[:, feature_dict[feature_type]] = curr_plate_features_df[feature_dict[feature_type]].astype(float)
        norm_cell_df.loc[norm_cell_df['Metadata_Plate'] == plate] = curr_plate_df
    return norm_cell_df

norm_cell_df = combined_cell_df_mitolyso_merged.copy()
print("Before conversion: AgeGroup dtype:", norm_cell_df["AgeGroup"].dtype)
print("Unique AgeGroup values:", norm_cell_df["AgeGroup"].unique())

norm_cell_df["AgeGroup"] = pd.to_numeric(norm_cell_df["AgeGroup"], errors='raise')

print("After conversion: AgeGroup dtype:", norm_cell_df["AgeGroup"].dtype)
print("Unique AgeGroup values after conversion:", norm_cell_df["AgeGroup"].unique())
#display(norm_cell_df["AgeGroup"].value_counts())

norm_cell_df_cell = apply_feature_normalization(norm_cell_df, cell_features, curr_plates).copy()
norm_cell_df_nuc = apply_feature_normalization(norm_cell_df_cell, nuc_features, curr_plates).copy()
norm_cell_df_mito = apply_feature_normalization(norm_cell_df_nuc, mito_features, curr_plates).copy()
norm_cell_df_mitolyso = apply_feature_normalization(norm_cell_df_mito, lyso_features, curr_plates)

#watch out - merged df might be clipping off all of the features

## Trying out the pivot and groupby functions


In [ ]:
feature_meas = "AreaShape_Area"
single_feat_df = make_single_feature_df(
    final_filtered_df, "AllGroups", feature_meas, "Replicate_Number"
)
single_feat_df["Replicate_Str"] = "R" + single_feat_df["Replicate_Number"].astype(str) 

grouped_by_reps_single_feat_df = single_feat_df.groupby(["AllGroups", "Replicate_Str"]).mean()
print(grouped_by_reps_single_feat_df.columns)
display(grouped_by_reps_single_feat_df)


single_feat_df_sat = make_single_feature_df(
    final_filtered_df_2, "AllGroups", feature_meas, "Replicate_Number"
)
single_feat_df_sat["Replicate_Str"] = "R" + single_feat_df_sat["Replicate_Number"].astype(str)

grouped_by_reps_single_feat_df_sat = single_feat_df_sat.groupby(
    ["AllGroups", "Replicate_Str"]
).mean()
print(grouped_by_reps_single_feat_df_sat.columns)
display(grouped_by_reps_single_feat_df_sat)

fig, axs = plt.subplots(1,2,sharey=True,figsize=(18,8))
sns.set_context("talk")
sns.barplot(data=single_feat_df, x="AllGroups", y=feature_meas, hue="Replicate_Str",ax=axs[0])
sns.barplot(data=single_feat_df_sat, x="AllGroups", y=feature_meas, hue="Replicate_Str",ax=axs[1])
plt.legend().remove()
plt.tight_layout()
plt.show()

In [ ]:
import operator
df_to_filter = saturated_cells

def get_mini_filtered_df(
    final_filtered_df,
    condition_col="",
    valueslist=[
        "ImageNumber",
        "TimepointName",
        "Metadata_WellRow",
        "Metadata_WellColumn",
        "Metadata_Field",
        "AllGroups",
        "Replicate_Number",
        "SerialPassage_BatchNumber",
        "AgeGroup",
        "PassageNumber",
        "Number_Object_Number",
        "AreaShape_Area",
        "Cell_Nuclei_Area_Ratio",
        "Children_Mitochondria_Count",
        "Children_Lysosomes_Count",
        "Image_Width_DAPI",
        "Image_URL_MitoTracker_MAX",
        "Image_FileName_MitoTracker_MAX",
        "Image_FileName_LAMP1_MAX",
    ],
    op=operator.le,
    condition_value = 10
):
    mini_df = final_filtered_df[valueslist]

    mini_df["Metadata_Rep_RowColField"] = (
        "R"
        + mini_df["Replicate_Number"].astype(str)
        + "_r"
        + mini_df["Metadata_WellRow"].astype(str)
        + "c"
        + mini_df["Metadata_WellColumn"].astype(str)
        + "f"
        + mini_df["Metadata_Field"].astype(str)
        + ""
    )

    # Now add the filter
    filter_mini_df = mini_df[op(mini_df[condition_col], condition_value)]
    filter_mini_df_sorted = filter_mini_df.sort_values(
        by=["AllGroups"], key=lambda x: x.map(passage_groups_sort_key)
    ).reset_index(drop=True)

    # reduce cols for readability
    filter_mini_df_display = filter_mini_df_sorted[
        ["AllGroups", "Replicate_Number", "Image_FileName_MitoTracker_MAX", condition_col]
    ]

    display(filter_mini_df_display)
    
    return filter_mini_df_sorted


def find_replicate_cp_output_folder(path):
    import re

    replicate_pattern = r"_rep0(\d{1})_"  # Matches "RX" where X is the replicate number (placeholder for now)
    match = re.search(replicate_pattern, path)
    if match:
        replicate = int(match.group(1))
    else:
        replicate = None
    return replicate


def pull_up_cp_segmentation_image(parent_dir="~/", img_filename="", replicate=0, group=""):
    # Loop over the plates
    # make sure the filename in the format of: "Image_FileName_MitoTracker_MAX"
    from matplotlib import image as mpimg
    from PIL import Image

    img_filename_noext = img_filename.split(".")[0]
    for root, dirs, files in os.walk(parent_dir):
        for filename in files:
            if (
                img_filename_noext in filename
                and filename.endswith(".png")
                and "active" in root
                and replicate == find_replicate_cp_output_folder(root)
            ):
                img_path = os.path.join(
                    root, img_filename_noext + ".png"
                )  # make the path
                try:
                    
                    img = Image.open(img_path)
                    fig = plt.figure(figsize=(8, 8))
                    plt.imshow(img)
                    plt.axis("off")  # Turn off axis labels for a cleaner image display
                    plt.title(f"R{replicate}, {filename}, {group}")
                    plt.show()
                    print(f"Segmented image url: {img_path}")
                    # img.show()

                except FileNotFoundError:
                    print(
                        f"Image file {img_path} not found. Please ensure 'your_image.png' exists."
                    )
            # else:
            #     print(filename, img_filename_noext)
mini_filter_df = get_mini_filtered_df(df_to_filter, condition_col="Children_Mitochondria_Count", op=operator.le, condition_value=100)
index_code = 101

# pull_up_cp_segmentation_image(
#     "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output",
#     "MAX_ch2-r06c10f06.tif",
#     6,
# )
pull_up_cp_segmentation_image(
    "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output",
    mini_filter_df["Image_FileName_LAMP1_MAX"][index_code],
    mini_filter_df["Replicate_Number"][index_code],
    mini_filter_df["AllGroups"][index_code]
)


def query_group_replicate_condition(df, group, replicate_number=0, condition_col="", op=operator.eq, value=None):
    """
    Filter df by group, replicate_number, and a condition using a passed operator.
    Example: op=operator.lt for '<', op=operator.gt for '>', op=operator.eq for '=='
    """
    mask = (
        (df["AllGroups"] == group) &
        (df["Replicate_Number"] == replicate_number) &
        (op(df[condition_col], value))
    )
    return df[mask]



display(query_group_replicate_condition(df_to_filter, "P29+", 5, "Children_Mitochondria_Count", operator.le, 100).shape)
#61,61,0
# NOTE there are definitley more cells in p23-25 group - size biasing potentially?

> NOTE:
> R4 appears to have used the wrong masks from cellprofiler!! nuclei look fine but cells are over the wrong image
> e.g Segmented image url: /mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/20241112_rep04_output/v5_active/MAX_ch2-r02c05f20.png

# It's plotting time


## Most interesting features so far
- Intensity_MassDisplacment_MitoTracker - increase
- Median Mitochondria Location CenterMass Intensity X
  - decrease,splits into bimodal
  - similar for y
  - AreaShape_Center_X and Center_Y also have similar pattern
  - And Location_Center
- Median mitochondria loaction max intensity
  - Same as CenterMass
- Mean Mitochondria Solidity - increase to p29
- Children Mitochondria Count - gradual increase
  - But scales with size - Density is not sig (slight increase)
  - Total mito area increases; complements this
- Mean Mito Centroid distance - increased (more peripheral)
  - But median is much more vairable between batches
- Mean Mito Trunks - decreased
- Median mito area per cell area - increased
### Lysosomes
- Total area also goes up
- rep3 looks like an outlier here
- Also has the location_ maxintensity change and go bimodal
- bimodal-looking intensity
### Nuclei
- Nuc area increases; scales with cell size increases
- solidity, extent down
- Nuc area ratio - slight increase


In [ ]:
def super_splitviolinplot_helper_singleplot(
    data_df,
    group_avg_df,
    ax,
    x_value,
    y_value,
    title,
    replicate_col_name,
    pairs=None,
    order=None,
    annotate=False,
    test=None,
    shapiro=True,
    show_test_on_plot=False,
):
    if pairs is None:
        pairs = getpairs(data_df, x_value, order=order)
    print(pairs)
    sns.violinplot(
        data=data_df,
        x=x_value,
        y=y_value,  # hue=x_value,
        # palette="Set2",
        split=True,  # using split violin plots - only one side, basically looks like a histogram
        inner="quart",
        color="gainsboro",
        #fill = True
        width=0.9,
        linewidth=1.5,
        order=order,
        ax=ax,
    )
    sns.swarmplot(
        data=group_avg_df,
        x=x_value,
        y=y_value,
        hue=replicate_col_name,
        order=order,
        palette="pastel",
        size=10,
        edgecolor="k",
        linewidth=1,
        dodge=True,
        ax=ax,
    )
    # draw a boxplot to show the mean line
    sns.boxplot(
        data=group_avg_df,
        x=x_value,
        y=y_value,
        showmeans=True,
        meanline=True,
        meanprops={"color": "dimgray", "ls": "-", "lw": 2.5},
        medianprops={"visible": False},
        whiskerprops={"visible": False},
        zorder=2,
        showfliers=False,
        showbox=False,
        showcaps=False,
        ax=ax,
    )
    ax.set_title(title)

    # axes[0].text(
    #     x=row[x_value],
    #     y=row[y_value],
    #     s=str(row["Shapiro_normality"]),
    #     color="black",
    #     fontsize=10,
    #     ha="center"
    # )
    # use pivot table to get the average values for each group
    if annotate and test is not None:
        group_avg_pivot_table = average_groups_pivot(
            group_avg_df, x_value, y_value, replicate_col_name
        )
        try:
            ax = annotate_pairs_with_calculated_pvalues(
                ax,
                group_avg_df,
                group_avg_pivot_table,
                x_value,
                y_value,
                replicate_col_name=replicate_col_name,
                test_name=test,
                order=order,
                plot="violinplot",
                show_test_name=show_test_on_plot,
            )
        except Exception as e:
            print(f"Error annotating with statistical test: {e}")
            # ax = annotate_with_anova_tukey(ax, pairs, group_avg_df_pivot, x_value, y_value, replicate_col_name=replicate_col_name, order=order, plot="violinplot")
        # elif test == "kruskal":
        #     ax = annotate_with_kruskal(
        #         ax,
        #         pairs,
        #         group_avg_pivot_table,
        #         x_value,
        #         y_value,
        #         order=order,
        #         replicate_col_name=replicate_col_name,
        #         plot="violinplot",
        #     )
        if shapiro:
            ax = annotate_legend_with_shapiro(ax, group_avg_df, replicate_col_name)

    return ax

def single_feature_super_splitviolinplot(
    data_df,
    x_value="AllGroups",
    y_value="Cell_AreaShape_Area",
    replicate_col_name="Replicate_Number",
    out_dir=Path(""),
    xtitle=None,
    ytitle=None,
    order=None,
    legend=True,
    annotate=False,
    test=None,
    show_hist=False,
    remove_outliers=False,
    ylim=None,
    reps_to_exclude=[],
    shapiro=True,
    show=True,
    context = "talk",
    figsize = (8,6),
    truncate_outliers = False
):
    """Make a superplot to do multiple comparisons for a feature between different conditions
    Args:
        data_df_1 (_type_): _description_
        group_avg_df_1 (_type_): _description_
        data_df_2 (_type_): _description_
        group_avg_df_2 (_type_): _description_
        x_value (str, optional): _description_. Defaults to "AllGroups".
        y_value (str, optional): _description_. Defaults to "Cell_AreaShape_Area".
        replicate_col_name (str, optional): _description_. Defaults to "Replicate_Number".
        csv_dir (str, optional): _description_. Defaults to "".
        xtitle (_type_, optional): _description_. Defaults to None.
        ytitle (_type_, optional): _description_. Defaults to None.
    """
    import matplotlib.lines as mlines
    from statannotations.Annotator import Annotator
    from statannotations.stats.StatTest import StatTest
    from pathlib import Path

    if order == None:
        order = get_all_group_order()
    pairs = getpairs(data_df, x_value, order=order)
    print(pairs)
    
    try:
        bottom_fence = None#np.percentile(data_df[y_value], 0.000001)
        top_fence = np.percentile(data_df[y_value], 99.99)
    except ValueError as e:
        print(e)
        top_fence = None
    if truncate_outliers:
        axlim = (bottom_fence, top_fence)
    else:
        axlim=(None,None)
    if show_hist:
        hist = sns.kdeplot(
            data_df, x=y_value, hue=replicate_col_name, palette="pastel"
        )
        plt.xlim(axlim)
        plt.savefig(f"{Path(out_dir, f"{y_value}_{replicate_col_name}_histogram")}.png")
        plt.show()
        plt.close()

        df_sorted = data_df.sort_values(
            by=[x_value], key=lambda x: x.map(passage_groups_sort_key)
        ).reset_index(drop=True)
        
        import kaleido

        hist2 = px.histogram(
            df_sorted,
            x=y_value,
            color=x_value,
            marginal="box",
            # histnorm='probability density',
            #range_x=(0, top_fence),
        )
        hist2.write_image(
            Path(out_dir, f"{y_value}_histogram.png"), scale=1.5
        )
        hist2.show()
    fig, ax = plt.subplots(figsize=figsize)
    sns.set_context(context=context, font_scale=1.2)
    sns.set_theme(style="ticks")
    
    # plt.style.use("ggplot")

    feature_df = make_single_feature_df(
        data_df, group=x_value, feature=y_value, replicates=replicate_col_name
    )
    if reps_to_exclude:
        feature_df = feature_df[~feature_df[replicate_col_name].isin(reps_to_exclude)]
        print(f"removing replicates: {reps_to_exclude}")
    
    if remove_outliers is True:
        feature_df = remove_outliers_iqr(feature_df)
        display(feature_df)
    group_avg_df = average_groups_by_plate(
        feature_df, x_value=x_value, y_value=y_value, replicates=replicate_col_name
    )
    
    #display(group_avg_df)

    ax = super_splitviolinplot_helper_singleplot(
        feature_df,
        group_avg_df,
        ax,
        x_value,
        y_value,
        title=" ",
        replicate_col_name=replicate_col_name,
        pairs=pairs,
        order=order,
        annotate=annotate,
        test=test,
        shapiro=False
    )

    if legend:
        if shapiro:
            group_avg_df_shapiro = apply_shapiro_wilk_test_to_df(
                group_avg_df,
                feature_meas=y_value,
                replicate_col_name="Replicate_Number",
                alpha=0.05,
            )
            #display(group_avg_df_shapiro)
            ax = annotate_legend_with_shapiro(ax, group_avg_df_shapiro, replicate_col_name)
    else:
        ax.legend_.remove()
    if ytitle is not None:
        ax.set_ylabel(ytitle)
    else:
        ax.set_ylabel(y_value.replace("_"," "))
    if xtitle is not None:
        ax.set_xlabel(xtitle)
    if ylim is None:
        ylim=axlim
    
    ax.set_ylim(ylim)
    plt.tight_layout()
    sns.despine()
    plt.savefig(os.path.join(out_dir, f"{y_value}_{test}.png"))
    if show:
        plt.show()


order = get_all_group_order()
feature_meas = "AreaShape_Area"
ylabel = None#"Mitochondrial Density Per Cell (relative to youngest passage)"#None#"Mitochondria per cell"
xlabel = "Age Groups"
group = "AllGroups"

pairs = getpairs(combined_cell_df_mitolyso, group, order)
this_df = combined_cell_df_mitolyso_merged.copy()
data_df = this_df[this_df["Nuclei_AreaShape_Area"] > 1000] # >3500]   #Potentially use a theshold - this is the trough of the nuc size peak at 0
#this_df = combined_cell_df_mitolyso_merged.copy()
pallete = "pastel"
remove_outliers = True
reps_to_exclude=[1]
plot_dir = "plots/notnorm" 
os.makedirs(plot_dir, exist_ok=True)

figsize = (12, 7)

single_feature_super_splitviolinplot(
    data_df,
    x_value=group,
    y_value=feature_meas,
    replicate_col_name="Replicate_Number",
    xtitle=xlabel,
    ytitle=ylabel,
    out_dir=plot_dir,
    annotate=True,
    order=order,
    test="games",
    reps_to_exclude=reps_to_exclude,
    show_hist=True,
    #remove_outliers=True,
    legend=True,
    context = "talk",
    figsize = figsize,
    truncate_outliers=True
)

#NOTE: R5 has smallest cells in p23-25, which is also highest mito density 

In [ ]:
#Make the one without annoations
single_feature_super_splitviolinplot(
    data_df,
    x_value=group,
    y_value=feature_meas,
    replicate_col_name="Replicate_Number",
    xtitle=xlabel,
    ytitle=ylabel,
    out_dir=plot_dir,
    annotate=False,
    order=order,
    test="None",
    reps_to_exclude=reps_to_exclude,
    show_hist=False,
    # remove_outliers=True,
    legend=True,
    context="talk",
    figsize=figsize,
    truncate_outliers=True,
)


In [ ]:
def make_a_shitton_of_plots(
    order = [],
    xlabel = "Age Groups",
    group = "AllGroups",
    pairs = getpairs(combined_cell_df_mitolyso, group, order),
    pallete = "pastel",
    remove_outliers = False,
    reps_to_exclude=[],
    norm = False
):
    if not order:
        order = get_all_group_order()
    if norm:
        this_df = norm_cell_df.copy()
        big_out_folder = (
            "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/plots/norm"
        )
    else:
        this_df = combined_cell_df_mitolyso_merged.copy()
        big_out_folder = (
            "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/plots/notnorm"
        )
    os.makedirs(big_out_folder, exist_ok=True)
    #iterate and make big ball of plots
    for compartment, feature_dict in zip(feature_names, feature_dicts):
        feature_folder = Path(big_out_folder, compartment)
        Path.mkdir(feature_folder, exist_ok=True)
        for feature_type, features in feature_dict.items():
            newfolder = Path(feature_folder,feature_type)
            Path.mkdir(newfolder, exist_ok=True)
            for feature in features:
                single_feature_super_splitviolinplot(
                    this_df,
                    x_value=group,
                    y_value=feature,
                    replicate_col_name="Replicate_Number",
                    xtitle=xlabel,
                    ytitle=None,
                    out_dir=newfolder,
                    annotate=True,
                    order=order,
                    test="tukey_v2",
                    reps_to_exclude=reps_to_exclude,
                    show_hist=False,
                    remove_outliers=remove_outliers,
                    show=False,
                    legend=True,
                    context = "talk",
                    figsize = (12,8)
                )
make_a_shitton_of_plots(reps_to_exclude=[],norm=False)
make_a_shitton_of_plots(reps_to_exclude=[], norm=True)

## Attempting PCA

In [ ]:
vars = mito_features["areashape"]
    

In [ ]:

from sklearn.decomposition import PCA
n_components = 2
pca = PCA(n_components)

x_df = this_df[vars.__add__
    (["AllGroups",
    "Replicate_Number","AreaShape_Area"])].dropna()
display(x_df)
components = pca.fit_transform(x_df[vars])
display(components)

fig = px.scatter(components, x=0, y=1, color=x_df["AllGroups"],symbol=x_df["Replicate_Number"])
total_var = pca.explained_variance_ratio_.sum() * 100

# labels = {str(i): f"PC {i + 1}" for i in range(n_components)}
# labels["color"] = "AllGroups"

# fig = px.scatter_matrix(
#     components,
#     color=x_df["AllGroups"],
#     dimensions=range(n_components),
#     labels=labels,
#     title=f"Total Explained Variance: {total_var:.2f}%",
# )
# fig.update_traces(diagonal_visible=False)
fig.show()

# labels = {
#     str(i): f"PC {i + 1} ({var:.1f}%)"
#     for i, var in enumerate(pca.explained_variance_ratio_ * 100)
# }

# fig = px.scatter_matrix(
#     components, labels=labels, dimensions=range(4), color=df["species"]
# )
# fig.update_traces(diagonal_visible=False)
# fig.show()


In [ ]:
from umap import UMAP
from sklearn.preprocessing import StandardScaler

scaled_data = StandardScaler().fit_transform(x_df[vars])
reducer = UMAP(random_state=42)
reducer.fit(scaled_data)

embedding = reducer.transform(scaled_data)
# Verify that the result of calling transform is
# idenitical to accessing the embedding_ attribute
assert np.all(embedding == reducer.embedding_)
print(embedding.shape)

import sklearn.cluster as cluster

import sklearn.metrics as metrics

kmeans_labels = cluster.KMeans(n_clusters=9).fit_predict(scaled_data)

"""
From https://umap-learn.readthedocs.io/en/latest/clustering.html: 
The next thing to be aware of is that when using UMAP for dimension reduction you will want to select different parameters 
than if you were using it for visualization. First of all we will want a larger n_neighbors value 
small values will focus more on very local structure and are more prone to producing fine grained cluster structure 
that may be more a result of patterns of noise in the data than actual clusters. 
In this case well double it from the default 15 up to 30. Second it is beneficial to set min_dist to a very low value. 
Since we actually want to pack points together densely (density is what we want after all) a low value will help,
as well as making cleaner separations between clusters.
In this case we will simply set min_dist to be 0.
"""
clusterable_embedding = UMAP(
    n_neighbors=30,
    min_dist=0.0,
    n_components=2,
    random_state=42,
).fit_transform(scaled_data)

#now plot the embeddings
fig = px.scatter(
    embedding,
    x=0,
    y=1,
    color=kmeans_labels,#x_df["Replicate_Number"],
    symbol=x_df["AllGroups"],
    size=x_df["AreaShape_Area"]
    #labels={"color": kmeans_labels},
)
fig.show()


labels = cluster.HDBSCAN(
    min_samples=10,
    min_cluster_size=500,
).fit_predict(clusterable_embedding)

fig = px.scatter(
    clusterable_embedding,
    x=0,
    y=1,
    color=labels,
    symbol=x_df["AllGroups"],  # x_df["Replicate_Number"],
    size=x_df["AreaShape_Area"],
    labels={"symbol" :x_df["AllGroups"]}#{"color": labels, }
)
fig.update_legends()
fig.show()




In [ ]:
#pairplot for funsies
# sns.pairplot(
#     combined_cell_df_mitolyso_merged,
#     hue="AllGroups",
#     vars=mito_features["radialdistribution"],
#     diag_kind="kde",
#     plot_kws={"alpha": 0.5},
# )
# plt.show()

In [ ]:
#Using one-way anova and Tukey's HSD to compare means of non normalized values
order = get_all_group_order()
feature_meas = "AreaShape_Area"
ylabel = "Area"


group = 'AllGroups'
replicates = 'Replicate_Number'
pairs = getpairs(combined_cell_df_mitolyso, group, order)
this_df = combined_cell_df_mitolyso_merged.copy()
pallete = "pastel"
remove_outliers = True

feature_df = make_single_feature_df(this_df, group=group, feature=feature_meas, replicates='Replicate_Number')
group_avg_df = average_groups_by_plate(feature_df, x_value=group, y_value=feature_meas, replicates='Replicate_Number')
group_avg_df_pivot = average_groups_pivot(group_avg_df, x_value=group, y_value=feature_meas, replicate_col_name='Replicate_Number')

if remove_outliers is True:
    feature_df = remove_outliers_iqr(feature_df)
    display(feature_df)

display(feature_df)
display(group_avg_df)
display(group_avg_df_pivot)

sns.set_theme(style="ticks")
#sns.set_context("notebook", font_scale=1.9)

plt.figure(figsize=(12, 8))
sns.set_context("talk", font_scale=0.5)
plt.figure(dpi=300)


sns.violinplot(data=feature_df, x=group,
            y=feature_meas,
            order=order,
            fill = False,
            color= 'gainsboro',
            cut=1,
            native_scale=True,
            linecolor='k',
            inner= None,
            #inner_kws=dict(box_width = 5)
            )

ax = sns.swarmplot(data=group_avg_df, x=group,
            y=feature_meas,
            hue = replicates,
            order=order,
            palette=pallete,
            size=10, 
            edgecolor="k", 
            linewidth=1,
            dodge=0.5)

#use a boxplot to draw the mean line - thinking outside the box :)
sns.boxplot(data = group_avg_df, x = group,
            y = feature_meas,
            showmeans=True,
            meanline=True,
            meanprops={'color': 'dimgray', 'ls': '-', 'lw': 2.5},
            medianprops={'visible': False},
            whiskerprops={'visible': False},
            zorder=1,
            showfliers=False,
            showbox=False,
            showcaps=False,
            ax = ax)

ax.legend_.remove()

sns.despine()
plt.gcf()#.set_size_inches(10, 6)
plt.xlabel(group)
if ylabel == None:
    plt.ylabel(feature_meas.replace('_', ' '))


from statannotations.Annotator import Annotator
from statannotations.stats.StatTest import StatTest

# Extract the data for each group

# Perform the one-way ANOVA test

# Print the results

pvalues,pairs = anova_with_tukey_posthoc(
    group_avg_df, x_value=group, y_value=feature_meas, display_results=True
)


annotator = Annotator(ax, pairs, data=group_avg_df_pivot, order=order)
annotator.configure(text_format='star', loc='inside', verbose = 2, hide_non_significant=True)
annotator.set_pvalues_and_annotate(pvalues)

plt.savefig("plots/new_plots/" + feature_meas + '_anova_superviolinplot.png', dpi=300)
plt.show()



## Summary Stats

In [ ]:
summary_outpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/Cell_Size_Data/summary_stats/"
feature_name = "AreaShape_Area"
feature_shortname = "Area"
included_cols = [
    "ImageNumber",
    "AreaShape_Area",
    "Nuclei_AreaShape_Area",
    "Cell_Nuclei_Area_Ratio",
    "Children_Lysosomes_Count",
    "Children_Mitochondria_Count",
]


def make_summary_stats_for_df_and_feature(
    df,
    x_value,
    feature,
    summary_outpath,
    df_tag="original",
    replicate_col_name="Replicate_Number",
    feature_name="AreaShape_Area",
    group_name="AllGroups",
    include_cols=[],
):
    try:
        table_csvname = f"{df_tag}_total_combined_{feature_name}_stats.csv"
        feature_csvname = f"{df_tag}_{feature_name}_by_{group_name}_stats.csv"
        agg_feature_csvname = f"{df_tag}_agg_{feature_name}_by_{group_name}_stats.csv"

        subfolder_name = f"{df_tag}_{feature_name}_summary_stats"
        parent_folder = Path(summary_outpath, subfolder_name)
        parent_folder.mkdir(exist_ok=True)

        if not include_cols:
            df_to_summarize = df
        else:
            df_to_summarize = df[include_cols]
        df_to_summarize.describe().to_csv(
            os.path.join(summary_outpath, subfolder_name, table_csvname)
        )
        group_averages = df.groupby(
            [x_value, replicate_col_name], as_index=False, observed=True
        )[feature]
        # Reset the index to get a clean DataFrame
        # average_df = group_averages.reset_index()
        avg_summary = group_averages.describe()
        avg_summary_sorted = avg_summary.sort_values(
            by=[x_value], key=lambda x: x.map(passage_groups_sort_key)
        ).reset_index(drop=True)
        avg_summary_sorted.to_csv(
            os.path.join(summary_outpath, subfolder_name, feature_csvname)
        )

        # do the agg by passage group only
        group_averages_agg = df.groupby([x_value], as_index=False, observed=True)[
            feature
        ]
        avg_agg_summary = group_averages_agg.describe()
        avg_agg_summary_sorted = avg_agg_summary.sort_values(
            by=[x_value], key=lambda x: x.map(passage_groups_sort_key)
        )
        avg_agg_summary_sorted.to_csv(
            os.path.join(summary_outpath, subfolder_name, agg_feature_csvname)
        )
        print(
            f"saved files {(table_csvname, feature_csvname, agg_feature_csvname)} to {summary_outpath}"
        )
        return True
    except ValueError as e:
        print(f"Could not make summary stats: {e}")
        return False


summary_test = make_summary_stats_for_df_and_feature(
    combined_cell_df_mitolyso_merged,
    "AllGroups",
    feature_name,
    summary_outpath,
    df_tag="original",
    feature_name=feature_shortname,
    include_cols=included_cols,
)


### To export the normalized csv:


In [ ]:
preprocessed_df = combined_cell_df_mitolyso_merged.copy()

preprocessed_df.to_csv(
    os.path.join(csvpath, "CellProfiler_features_preprocessed.csv"), index=False
)

norm_cell_df_mitolyso.to_csv(
    os.path.join(csvpath, "Norm_CellProfiler_features_preprocessed.csv"),
    index=False,
)